In [ ]:
import torch
from IPython.display import display
from PIL import Image
from mvadapter.utils import (
    draw_patches,
    get_heatmap
)
from mvadapter.models.attention_util import (
    extract_attention_map_from_query_patch, 
    extract_attention_map_from_query_column
)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
filename = "dino"

In [ ]:
generated_mv = f"output/{filename}/mvimages.png"
image = Image.open(generated_mv)
display(image)

In [ ]:
W, H = image.size
N = W//H
images = [image.crop((i*H, 0, (i+1)*H, H)) for i in range(N)]

Reference Image Cross Attention

In [ ]:
attn_map = f"output/{filename}/cross_attention.pt"
cross_attn_weights = torch.load(attn_map)

In [ ]:
v = 2
p = 319

In [ ]:
display(draw_patches(images[v], num_patches=(24, 24), highlight_index=p, line_width=1).resize((256, 256)))

In [ ]:
heatmaps=[]
for step in range(len(cross_attn_weights)):
    attention_map = extract_attention_map_from_query_patch(cross_attn_weights[step], selected_view=v, selected_patch=p)
    heatmap = get_heatmap(attention_map, ref_image_path= f"output/{filename}/reference.png")
    heatmaps.append(heatmap)
black_frame = Image.new("RGB", (heatmaps[0].width, heatmaps[0].height), "black")
black_frame.save(f"output/{filename}/cross_attn-{v}-{p}.gif", save_all=True, append_images=heatmaps[:], duration=10, loop=0)

Multiview Self Attention

In [ ]:
self_attn_map = f"output/{filename}/self_attention.pt"
self_attn_weights = torch.load(self_attn_map)

In [ ]:
v_self = 1
column = 8

In [ ]:
display(draw_patches(images[v_self], num_patches=(24, 24), highlight_column=column, line_width=1).resize((256, 256)))

In [ ]:
heatmaps=[]
for step in range(len(self_attn_weights)):
    attention_map = extract_attention_map_from_query_column(self_attn_weights[step], selected_view=v_self, selected_column=column)
    heatmap = get_heatmap(attention_map, ref_image_path= f"output/{filename}/mvimages.png")
    #display(heatmap)
    heatmaps.append(heatmap)
black_frame = Image.new("RGB", (heatmaps[0].width, heatmaps[0].height), "black")
black_frame.save(f"output/{filename}/self_attn-{v_self}-{column}.gif", save_all=True, append_images=heatmaps[:], duration=10, loop=0)